# 02 Load FinanceBench Sample

## Σκοπός
Σε αυτό το notebook:

- φορτώνουμε το FinanceBench sample
- εξετάζουμε τις στήλες του dataset
- κάνουμε βασικό data inspection
- εντοπίζουμε πώς συνδέονται οι ερωτήσεις με τα reports/PDFs
- προετοιμάζουμε ένα καθαρό working dataframe για τα επόμενα βήματα

In [26]:
from pathlib import Path
import warnings
import json

import pandas as pd
import numpy as np

In [27]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PDFS_DIR = RAW_DIR / "pdfs"
INTERIM_DIR = DATA_DIR / "interim"

QUESTIONS_PATH = RAW_DIR / "financebench_open_source.jsonl"
DOC_INFO_PATH = RAW_DIR / "financebench_document_information.jsonl"

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PDFS_DIR:", PDFS_DIR)
print("QUESTIONS_PATH exists:", QUESTIONS_PATH.exists())
print("DOC_INFO_PATH exists:", DOC_INFO_PATH.exists())

BASE_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag
RAW_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw
PDFS_DIR: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs
QUESTIONS_PATH exists: True
DOC_INFO_PATH exists: True


In [28]:
df_questions = pd.read_json(QUESTIONS_PATH, lines=True)
df_docs = pd.read_json(DOC_INFO_PATH, lines=True)

print("df_questions shape:", df_questions.shape)
print("df_docs shape:", df_docs.shape)

df_questions shape: (150, 11)
df_docs shape: (361, 6)


In [29]:
print("Questions columns:")
for col in df_questions.columns:
    print("-", col)

print("\nDocument info columns:")
for col in df_docs.columns:
    print("-", col)

Questions columns:
- financebench_id
- company
- doc_name
- question_type
- question_reasoning
- domain_question_num
- question
- answer
- justification
- dataset_subset_label
- evidence

Document info columns:
- doc_name
- company
- gics_sector
- doc_type
- doc_period
- doc_link


In [30]:
display(df_questions.head(2))
display(df_docs.head(2))

,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,$1577.00,"The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Statement of Cash Flow s Years ended December 31 (Millions) 2018 2017 2016 Cash Flows from Operating Activ...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,$8.70,"The metric ppne, net was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Property, plant and equipment â net.",OPEN_SOURCE,"[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Balance Shee t At December 31 December 31, December 31, (Dollars in millions, except per share amount) 2..."


,doc_name,company,gics_sector,doc_type,doc_period,doc_link
0,3M_2015_10K,3M,Industrials,10k,2015,https://investors.3m.com/financials/sec-filings/content/0001558370-16-003162/0001558370-16-003162.pdf
1,3M_2016_10K,3M,Industrials,10k,2016,https://investors.3m.com/financials/sec-filings/content/0001558370-17-000479/0001558370-17-000479.pdf


In [31]:
df_full = pd.merge(
    df_questions,
    df_docs,
    on="doc_name",
    how="left",
    suffixes=("", "_doc")
)

print("df_full shape:", df_full.shape)
df_full.head(3)

df_full shape: (150, 16)


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,company_doc,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,$1577.00,"The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Statement of Cash Flow s Years ended December 31 (Millions) 2018 2017 2016 Cash Flows from Operating Activ...,3M,Industrials,10k,2018,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,$8.70,"The metric ppne, net was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Property, plant and equipment â net.",OPEN_SOURCE,"[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Balance Shee t At December 31 December 31, December 31, (Dollars in millions, except per share amount) 2...",3M,Industrials,10k,2018,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY2022 data?,"No, the company is managing its CAPEX and Fixed Assets pretty efficiently, which is evident from below key metrics:\nCAPEX/Revenue Ratio: 5.1%\nFixed assets/Total Assets: 20%\nReturn on Assets= 12.4%",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=Net Income/Total Assets,OPEN_SOURCE,"[{'evidence_text': '3M Company and Subsidiaries Consolidated Statement of Income Years ended December 31 (Millions, except per share amounts) 2022 2021 2020 Net sales $ 34,229 $ 35,355 $ 32,184', ...",3M,Industrials,10k,2022,https://investors.3m.com/financials/sec-filings/content/0000066740-23-000014/0000066740-23-000014.pdf


In [32]:
merge_summary = {
    "question_rows": len(df_questions),
    "doc_rows": len(df_docs),
    "merged_rows": len(df_full),
    "missing_doc_metadata_rows": int(df_full["doc_type"].isna().sum()) if "doc_type" in df_full.columns else None,
    "unique_doc_names_questions": df_questions["doc_name"].nunique(),
    "unique_doc_names_docs": df_docs["doc_name"].nunique(),
}

pd.DataFrame([merge_summary])

,question_rows,doc_rows,merged_rows,missing_doc_metadata_rows,unique_doc_names_questions,unique_doc_names_docs
0,150,361,150,0,84,360


In [33]:
missing_df = pd.DataFrame({
    "column": df_full.columns,
    "missing_count": df_full.isna().sum().values,
    "missing_pct": (df_full.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_df

,column,missing_count,missing_pct
5,domain_question_num,100,66.67
4,question_reasoning,50,33.33
8,justification,50,33.33
0,financebench_id,0,0.00
3,question_type,0,0.00
2,doc_name,0,0.00
6,question,0,0.00
1,company,0,0.00
7,answer,0,0.00
9,dataset_subset_label,0,0.00


In [34]:
record = df_full.iloc[0].to_dict()

for k, v in record.items():
    print(f"{k}: {v}\n")

financebench_id: financebench_id_03029

company: 3M

doc_name: 3M_2018_10K

question_type: metrics-generated

question_reasoning: Information extraction

domain_question_num: None

question: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

answer: $1577.00

justification: The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).

dataset_subset_label: OPEN_SOURCE

evidence: [{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncontrolling interest to 

In [35]:
print(type(df_full.loc[0, "evidence"]))
print(df_full.loc[0, "evidence"])

<class 'list'>
[{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncontrolling interest to net cash\nprovided by operating activities\n \n \n \n \n \n \n \nDepreciation and amortization\n \n \n1,488 \n \n1,544 \n \n1,474 \nCompany pension and postretirement contributions\n \n \n(370) \n \n(967) \n \n(383) \nCompany pension and postretirement expense\n \n \n410 \n \n334 \n \n250 \nStock-based compensation expense\n \n \n302 \n \n324 \n \n298 \nGain on sale of businesses\n \n \n(545) \n \n(586) \n \n(111) \nDeferred income taxes\n \n \n(57) \n \n107 \n \n 7 \nChanges in assets and liabilities\n \n \n \n \n \n \n \nAccounts receivable\n \n \n(305) \n \n(245) \n \n(313) \nInvento

In [36]:
first_evidence = df_full.loc[0, "evidence"][0] if len(df_full.loc[0, "evidence"]) > 0 else {}
first_evidence

{'evidence_text': 'Table of Contents \n3M Company and Subsidiaries\nConsolidated Statement of Cash Flow s\nYears ended December 31\n \n(Millions)\n \n2018\n \n2017\n \n2016\n \nCash Flows from Operating Activities\n \n \n \n \n \n \n \nNet income including noncontrolling interest\n \n$\n5,363 \n$\n4,869 \n$\n5,058 \nAdjustments to reconcile net income including noncontrolling interest to net cash\nprovided by operating activities\n \n \n \n \n \n \n \nDepreciation and amortization\n \n \n1,488 \n \n1,544 \n \n1,474 \nCompany pension and postretirement contributions\n \n \n(370) \n \n(967) \n \n(383) \nCompany pension and postretirement expense\n \n \n410 \n \n334 \n \n250 \nStock-based compensation expense\n \n \n302 \n \n324 \n \n298 \nGain on sale of businesses\n \n \n(545) \n \n(586) \n \n(111) \nDeferred income taxes\n \n \n(57) \n \n107 \n \n 7 \nChanges in assets and liabilities\n \n \n \n \n \n \n \nAccounts receivable\n \n \n(305) \n \n(245) \n \n(313) \nInventories\n \n \n(509

In [37]:
stats = {
    "n_questions": len(df_full),
    "n_unique_companies": df_full["company"].nunique(),
    "n_unique_docs": df_full["doc_name"].nunique(),
    "question_types": df_full["question_type"].nunique() if "question_type" in df_full.columns else None,
    "reasoning_types": df_full["question_reasoning"].nunique() if "question_reasoning" in df_full.columns else None,
}

pd.DataFrame([stats])

,n_questions,n_unique_companies,n_unique_docs,question_types,reasoning_types
0,150,32,84,3,9


In [38]:
if "question_type" in df_full.columns:
    display(df_full["question_type"].value_counts(dropna=False).to_frame("count"))

if "question_reasoning" in df_full.columns:
    display(df_full["question_reasoning"].value_counts(dropna=False).head(20).to_frame("count"))

if "doc_type" in df_full.columns:
    display(df_full["doc_type"].value_counts(dropna=False).to_frame("count"))

,count
question_type,
metrics-generated,50
domain-relevant,50
novel-generated,50


,count
question_reasoning,
None,50
Numerical reasoning,43
Information extraction,31
Numerical reasoning OR Logical reasoning,6
Logical reasoning (based on numerical reasoning),5
Logical reasoning (based on numerical reasoning) OR Logical reasoning,5
Logical reasoning (based on numerical reasoning) OR Numerical reasoning OR Logical reasoning,4
Numerical reasoning OR information extraction,4
Information extraction OR Logical reasoning OR,1


,count
doc_type,
10k,112
10q,15
Earnings,14
8k,9


In [39]:
pdf_files = sorted(PDFS_DIR.glob("*.pdf"))

pdf_inventory = pd.DataFrame({
    "pdf_filename": [p.name for p in pdf_files],
    "pdf_stem": [p.stem for p in pdf_files],
    "pdf_path": [str(p) for p in pdf_files]
})

print("Local PDFs found:", len(pdf_inventory))
pdf_inventory.head()

Local PDFs found: 88


,pdf_filename,pdf_stem,pdf_path
0,3M_2018_10K.pdf,3M_2018_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf
1,3M_2022_10K.pdf,3M_2022_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2022_10K.pdf
2,3M_2023Q2_10Q.pdf,3M_2023Q2_10Q,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2023Q2_10Q.pdf
3,ACTIVISIONBLIZZARD_2019_10K.pdf,ACTIVISIONBLIZZARD_2019_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\ACTIVISIONBLIZZARD_2019_10K.pdf
4,ADOBE_2015_10K.pdf,ADOBE_2015_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\ADOBE_2015_10K.pdf


In [40]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

df_full["normalized_doc_name"] = df_full["doc_name"].apply(normalize_text)
pdf_inventory["normalized_pdf_stem"] = pdf_inventory["pdf_stem"].apply(normalize_text)

display(df_full[["doc_name", "normalized_doc_name"]].head())
display(pdf_inventory.head())

,doc_name,normalized_doc_name
0,3M_2018_10K,3m 2018 10k
1,3M_2018_10K,3m 2018 10k
2,3M_2022_10K,3m 2022 10k
3,3M_2022_10K,3m 2022 10k
4,3M_2022_10K,3m 2022 10k


,pdf_filename,pdf_stem,pdf_path,normalized_pdf_stem
0,3M_2018_10K.pdf,3M_2018_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf,3m 2018 10k
1,3M_2022_10K.pdf,3M_2022_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2022_10K.pdf,3m 2022 10k
2,3M_2023Q2_10Q.pdf,3M_2023Q2_10Q,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2023Q2_10Q.pdf,3m 2023q2 10q
3,ACTIVISIONBLIZZARD_2019_10K.pdf,ACTIVISIONBLIZZARD_2019_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\ACTIVISIONBLIZZARD_2019_10K.pdf,activisionblizzard 2019 10k
4,ADOBE_2015_10K.pdf,ADOBE_2015_10K,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\ADOBE_2015_10K.pdf,adobe 2015 10k


In [41]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

In [42]:
df_matched = df_full.merge(
    pdf_inventory,
    left_on="normalized_doc_name",
    right_on="normalized_pdf_stem",
    how="left"
)

match_summary = {
    "total_rows": len(df_matched),
    "matched_rows": int(df_matched["pdf_filename"].notna().sum()),
    "unmatched_rows": int(df_matched["pdf_filename"].isna().sum()),
    "unique_docs_in_dataset": df_matched["doc_name"].nunique(),
    "unique_local_pdfs": len(pdf_inventory),
    "matched_unique_docs": df_matched.loc[df_matched["pdf_filename"].notna(), "doc_name"].nunique()
}

pd.DataFrame([match_summary])

,total_rows,matched_rows,unmatched_rows,unique_docs_in_dataset,unique_local_pdfs,matched_unique_docs
0,150,150,0,84,88,84


In [43]:
unmatched_docs = (
    df_matched.loc[df_matched["pdf_filename"].isna(), ["doc_name", "company", "doc_type", "doc_period"]]
    .drop_duplicates()
    .sort_values(["company", "doc_name"])
)

print("Unmatched unique docs:", len(unmatched_docs))
unmatched_docs.head(20)

Unmatched unique docs: 0


,doc_name,company,doc_type,doc_period


In [44]:
company_counts = (
    df_matched["company"]
    .value_counts()
    .reset_index()
)
company_counts.columns = ["company", "question_count"]

doc_counts = (
    df_matched["doc_name"]
    .value_counts()
    .reset_index()
)
doc_counts.columns = ["doc_name", "question_count"]

display(company_counts.head(15))
display(doc_counts.head(15))

,company,question_count
0,PepsiCo,11
1,Amcor,9
2,Johnson & Johnson,9
3,3M,8
4,Boeing,8
5,Best Buy,8
6,AMD,8
7,American Express,7
8,MGM Resorts,7
9,Pfizer,6


,doc_name,question_count
0,AMERICANEXPRESS_2022_10K,7
1,AMD_2022_10K,7
2,BOEING_2022_10K,7
3,PEPSICO_2022_10K,5
4,AMCOR_2023_10K,4
5,ULTABEAUTY_2023Q4_EARNINGS,4
6,3M_2023Q2_10Q,3
7,JOHNSON_JOHNSON_2022_10K,3
8,PFIZER_2021_10K,3
9,AES_2022_10K,3


In [45]:
for col in ["question", "answer", "justification"]:
    if col in df_matched.columns:
        lengths = df_matched[col].fillna("").astype(str).str.len()
        print(f"\nColumn: {col}")
        print(lengths.describe())


Column: question
count    150.000000
mean     161.093333
std       96.894902
min       44.000000
25%       80.000000
50%      137.500000
75%      210.750000
max      592.000000
Name: question, dtype: float64

Column: answer
count    150.000000
mean      78.186667
std       96.555745
min        1.000000
25%        6.250000
50%       50.500000
75%      108.750000
max      609.000000
Name: answer, dtype: float64

Column: justification
count    150.000000
mean     150.453333
std      171.245363
min        0.000000
25%        0.000000
50%      100.500000
75%      219.500000
max      703.000000
Name: justification, dtype: float64


In [46]:
working_df = df_matched.copy().reset_index(drop=True)
working_df["row_id"] = working_df.index

priority_cols = [
    "row_id",
    "financebench_id",
    "question",
    "answer",
    "company",
    "doc_name",
    "doc_type",
    "doc_period",
    "pdf_filename",
    "pdf_path",
    "question_type",
    "question_reasoning",
    "justification",
    "evidence"
]

existing_priority_cols = [c for c in priority_cols if c in working_df.columns]
remaining_cols = [c for c in working_df.columns if c not in existing_priority_cols]

working_df = working_df[existing_priority_cols + remaining_cols]
working_df.head()

,row_id,financebench_id,question,answer,company,doc_name,doc_type,doc_period,pdf_filename,pdf_path,question_type,question_reasoning,justification,evidence,domain_question_num,dataset_subset_label,company_doc,gics_sector,doc_link,normalized_doc_name,pdf_stem,normalized_pdf_stem
0,0,financebench_id_03029,What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.,$1577.00,3M,3M_2018_10K,10k,2018,3M_2018_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf,metrics-generated,Information extraction,"The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).",[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Statement of Cash Flow s Years ended December 31 (Millions) 2018 2017 2016 Cash Flows from Operating Activ...,None,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf,3m 2018 10k,3M_2018_10K,3m 2018 10k
1,1,financebench_id_04672,Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer ...,$8.70,3M,3M_2018_10K,10k,2018,3M_2018_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2018_10K.pdf,metrics-generated,Information extraction,"The metric ppne, net was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Property, plant and equipment â net.","[{'evidence_text': 'Table of Contents 3M Company and Subsidiaries Consolidated Balance Shee t At December 31 December 31, December 31, (Dollars in millions, except per share amount) 2...",None,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0001558370-19-000470/0001558370-19-000470.pdf,3m 2018 10k,3M_2018_10K,3m 2018 10k
2,2,financebench_id_00499,Is 3M a capital-intensive business based on FY2022 data?,"No, the company is managing its CAPEX and Fixed Assets pretty efficiently, which is evident from below key metrics:\nCAPEX/Revenue Ratio: 5.1%\nFixed assets/Total Assets: 20%\nReturn on Assets= 12.4%",3M,3M_2022_10K,10k,2022,3M_2022_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2022_10K.pdf,domain-relevant,Logical reasoning (based on numerical reasoning),CAPEX/Revenue\nFixed Assets/Total Assets\nROA=Net Income/Total Assets,"[{'evidence_text': '3M Company and Subsidiaries Consolidated Statement of Income Years ended December 31 (Millions, except per share amounts) 2022 2021 2020 Net sales $ 34,229 $ 35,355 $ 32,184', ...",dg06,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0000066740-23-000014/0000066740-23-000014.pdf,3m 2022 10k,3M_2022_10K,3m 2022 10k
3,3,financebench_id_01226,"What drove operating margin change as of FY2022 for 3M? If operating margin is not a useful metric for a company like this, then please state that and explain why.","Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due to: \n-Decrease in gross Margin\n-mostly one-off charges including Combat Arms Earplugs litigation, impairment related to exit...",3M,3M_2022_10K,10k,2022,3M_2022_10K.pdf,C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\raw\pdfs\3M_2022_10K.pdf,domain-relevant,Logical reasoning (based on numerical reasoning) OR Numerical reasoning OR Logical reasoning,None,"[{'evidence_text': 'SG&A, measured as a percent of sales, increased in 2022 when compared to the same period last year. SG&A was impacted by increased special item costs for significant litigation...",dg17,OPEN_SOURCE,3M,Industrials,https://investors.3m.com/financials/sec-filings/content/0000066740-23-000014/0000066740-23-000014.pdf,3m 2

In [47]:
working_csv_path = INTERIM_DIR / "financebench_open_source_working.csv"
working_parquet_path = INTERIM_DIR / "financebench_open_source_working.parquet"

working_df.to_csv(working_csv_path, index=False)
print("Saved CSV to:", working_csv_path)

try:
    working_df.to_parquet(working_parquet_path, index=False)
    print("Saved Parquet to:", working_parquet_path)
except ImportError as e:
    print("Parquet save skipped.")
    print("Reason:", e)
    print("Install pyarrow with: pip install pyarrow")

Saved CSV to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\financebench_open_source_working.csv
Saved Parquet to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\financebench_open_source_working.parquet


In [48]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

Saved unmatched docs to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\financebench_unmatched_docs.csv


In [49]:
output_path = DATA_DIR / "interim" / "financebench_sample_working.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

working_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\financebench_sample_working.csv


In [50]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

Saved unmatched docs to: C:\Users\ntheo\Desktop\Repositories\Financial-RAG\thesis-rag\data\interim\financebench_unmatched_docs.csv


## Συμπέρασμα

Σε αυτό το notebook:

- φορτώσαμε τα δύο JSONL αρχεία του FinanceBench sample
- ενώσαμε questions και document metadata με βάση το `doc_name`
- ελέγξαμε τα τοπικά PDFs
- κάναμε πρώτη αντιστοίχιση dataset documents ↔ local files
- αποθηκεύσαμε working dataset για τα επόμενα στάδια

Το επόμενο notebook είναι το `03_parse_pdfs_with_docling.ipynb`.